# Track B: BRCA1 Mutation Walk -- Analysis (4-Model Comparison)

**Purpose**: Load cached embeddings from all four models, compute Lipschitz profiles,
and generate comparison visualizations.

## Models Compared

| Model | Architecture | Tokenization | Type | Role |
|-------|-------------|-------------|------|------|
| **DNABERT-2** (117M) | BERT Transformer | BPE | Discrete | "Jagged staircase" baseline |
| **Nucleotide Transformer** (500M) | BERT Transformer | 6-mer | Discrete | Second Transformer flavor |
| **Evo 2** (7B) | StripedHyena | Single-char | Hybrid (SSM + Attn) | Discrete tokenization + SSM routing |
| **Caduceus** (7.7M) | Mamba SSM | Single-char (RC-eq) | Continuous (SSM) | "Smooth ribbon" hypothesis |

## The Spectrum
```
Pure Transformer          Hybrid              Pure SSM
(Discrete Tokens)     (Discrete + SSM)     (Continuous)
 DNABERT-2  NT         Evo 2              Caduceus
   BPE     6-mer     single-char         single-char (RC)
```

If the Geometric Tax thesis holds, we expect Lipschitz roughness to decrease
left-to-right: Transformers show spikes, Caduceus shows smooth flow,
and Evo 2 falls somewhere in between.

## Prerequisites
Run all four embedding notebooks first:
- `Track_B_Caduceus.ipynb`
- `Track_B_Evo2.ipynb`
- `Track_B_DNABERT2.ipynb`
- `Track_B_NucleotideTransformer.ipynb`

All cache to: `./results/interpolation_track_b/cache/`

In [ ]:
# Cell: Configuration & Load Cached Embeddings

import os
import numpy as np
import torch
from scipy.spatial.distance import cosine

OUTPUT_BASE = './results/interpolation_track_b/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR   = OUTPUT_BASE + 'cache'

os.makedirs(RESULTS_DIR, exist_ok=True)

# --- Load mutation walk metadata ---
walk_data = np.load(f'{CACHE_DIR}/brca1_mutation_walk.npz', allow_pickle=True)
mutation_walk = walk_data['walk']
walk_positions = walk_data['positions']
pathogenic_step = int(walk_data['pathogenic_step'])

n_steps = len(mutation_walk)
print(f'Mutation walk: {n_steps} steps ({n_steps - 1} mutations)')
print(f'Pathogenic C61G at step {pathogenic_step}')

# --- Define model registry ---
MODEL_REGISTRY = {
    'DNABERT-2': {
        'cache': f'{CACHE_DIR}/dnabert2_brca1_walk.npy',
        'arch': 'Transformer (BPE)',
        'color': '#2563EB',    # blue
        'type': 'discrete',
        'notebook': 'Track_B_DNABERT2.ipynb',
    },
    'Nucleotide Transformer': {
        'cache': f'{CACHE_DIR}/nucleotide_transformer_brca1_walk.npy',
        'arch': 'Transformer (6-mer)',
        'color': '#7C3AED',    # purple
        'type': 'discrete',
        'notebook': 'Track_B_NucleotideTransformer.ipynb',
    },
    'Evo 2': {
        'cache': f'{CACHE_DIR}/evo2_brca1_walk.npy',
        'arch': 'StripedHyena (hybrid)',
        'color': '#DC2626',    # red
        'type': 'hybrid',
        'notebook': 'Track_B_Evo2.ipynb',
    },
    'Caduceus': {
        'cache': f'{CACHE_DIR}/caduceus_brca1_walk.npy',
        'arch': 'Mamba SSM (RC-eq)',
        'color': '#16A34A',    # green
        'type': 'continuous',
        'notebook': 'Track_B_Caduceus.ipynb',
    },
}

# --- Load all available embeddings ---
embeddings = {}
missing = []

for name, info in MODEL_REGISTRY.items():
    if os.path.exists(info['cache']):
        emb = np.load(info['cache'])
        embeddings[name] = emb
        print(f'  {name}: {emb.shape} loaded')
    else:
        missing.append(f'{name} (run {info["notebook"]} first)')

if missing:
    print(f'\nWARNING: Missing embeddings:')
    for m in missing:
        print(f'  - {m}')
    print('\nProceeding with available models...')
else:
    print(f'\nAll {len(MODEL_REGISTRY)} models loaded.')

# Verify step counts
for name, emb in embeddings.items():
    assert len(emb) == n_steps, (
        f'{name} has {len(emb)} embeddings but walk has {n_steps} steps')
print('All embedding counts match walk length.')

In [ ]:
# Cell: Compute Lipschitz Profiles for All Models

def compute_lipschitz_l2(embs):
    '''L2-based local Lipschitz constant between consecutive steps.'''
    n = len(embs)
    lip = np.zeros(n - 1)
    for i in range(n - 1):
        lip[i] = np.linalg.norm(embs[i + 1] - embs[i])
    return lip


def compute_cosine_lipschitz(embs):
    '''Cosine-based local Lipschitz (dimension-invariant).'''
    n = len(embs)
    lip = np.zeros(n - 1)
    for i in range(n - 1):
        d = cosine(embs[i], embs[i + 1])
        lip[i] = d if np.isfinite(d) else 0.0
    return lip


def compute_cosine_from_start(embs):
    '''Cosine distance of each step from wildtype.'''
    start = embs[0]
    return np.array([
        cosine(start, e) if np.isfinite(cosine(start, e)) else 0.0
        for e in embs
    ])


# Compute for all loaded models
lipschitz = {}
cosine_drift = {}

print('=' * 70)
print('LIPSCHITZ PROFILE SUMMARY')
print('=' * 70)

for name, emb in embeddings.items():
    lip_l2 = compute_lipschitz_l2(emb)
    lip_cos = compute_cosine_lipschitz(emb)
    cos_d = compute_cosine_from_start(emb)

    lipschitz[name] = lip_cos  # Use cosine for cross-model comparison
    cosine_drift[name] = cos_d

    threshold = lip_cos.mean() + 2 * lip_cos.std()
    n_spikes = int((lip_cos > threshold).sum())

    print(f'\n{name} (d={emb.shape[1]}, type={MODEL_REGISTRY[name]["type"]}):')
    print(f'  Cosine Lip: mean={lip_cos.mean():.6f}, max={lip_cos.max():.6f}, '
          f'std={lip_cos.std():.6f}')
    print(f'  Smoothness (mean/max): {lip_cos.mean() / (lip_cos.max() + 1e-8):.4f}')
    print(f'  Spikes (>2*std): {n_spikes}')

    if pathogenic_step is not None and pathogenic_step <= len(lip_cos):
        ps_val = lip_cos[pathogenic_step - 1]
        is_spike = ps_val > threshold
        print(f'  C61G Lipschitz: {ps_val:.6f} {"(SPIKE)" if is_spike else ""}')

# Save profiles
save_dict = {}
for name in embeddings:
    safe = name.lower().replace(' ', '_').replace('-', '_')
    save_dict[f'{safe}_lip_cos'] = lipschitz[name]
    save_dict[f'{safe}_cos_drift'] = cosine_drift[name]
np.savez(f'{CACHE_DIR}/lipschitz_profiles_v3_4model.npz', **save_dict)
print(f'\nProfiles saved to cache')

In [ ]:
# Cell: Export per-step Lipschitz profiles to CSV
# Used by figs/figure1_ground_truth.py for Panel A (Track B)

import pandas as pd

csv_rows = []
for name in embeddings:
    lip = lipschitz[name]
    norm_steps = np.linspace(0, 1, len(lip))
    for step_idx in range(len(lip)):
        csv_rows.append({
            'model': name,
            'architecture': MODEL_REGISTRY[name]['arch'],
            'type': MODEL_REGISTRY[name]['type'],
            'step': step_idx,
            'normalized_step': norm_steps[step_idx],
            'cosine_lipschitz': lip[step_idx],
        })

df_lip = pd.DataFrame(csv_rows)
lip_csv_path = f'{RESULTS_DIR}/track_b_lipschitz.csv'
df_lip.to_csv(lip_csv_path, index=False)
print(f'Per-step Lipschitz CSV saved: {lip_csv_path}')
print(f'  {len(df_lip)} rows ({len(embeddings)} models x {len(df_lip)//len(embeddings)} steps)')


In [ ]:
# Cell: Main Visualization -- 4-Model Grid (2x4: cosine drift + Lipschitz per model)

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

plt.rcParams.update({'font.size': 11})

n_models = len(embeddings)
n_walk = len(mutation_walk)
steps = np.arange(n_walk)
lip_steps = np.arange(1, n_walk)

fig, axes = plt.subplots(n_models, 3, figsize=(20, 5 * n_models))
if n_models == 1:
    axes = axes[np.newaxis, :]

for row, name in enumerate(embeddings):
    emb = embeddings[name]
    lip = lipschitz[name]
    cos_d = cosine_drift[name]
    color = MODEL_REGISTRY[name]['color']
    arch = MODEL_REGISTRY[name]['arch']

    # --- Panel A: Cosine distance from wildtype ---
    ax = axes[row, 0]
    ax.plot(steps, cos_d, color=color, linewidth=2)
    ax.fill_between(steps, cos_d, alpha=0.15, color=color)
    if pathogenic_step is not None:
        ax.axvline(pathogenic_step, color='red', linestyle='--', alpha=0.6,
                   label=f'C61G (step {pathogenic_step})')
    ax.set_xlabel('Mutation Walk Step')
    ax.set_ylabel('Cosine Distance from Wildtype')
    ax.set_title(f'{name} -- Cosine Distance', fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.2)

    # --- Panel B: Lipschitz profile ---
    ax = axes[row, 1]
    ax.plot(lip_steps, lip, color=color, linewidth=1.5, alpha=0.8)
    if pathogenic_step is not None and pathogenic_step <= len(lip):
        ax.scatter([pathogenic_step], [lip[pathogenic_step - 1]],
                   color='red', s=80, zorder=10, label='C61G')
    ax.axhline(lip.mean(), color='gray', linestyle='-', alpha=0.5, linewidth=1)
    ax.axhline(lip.mean() + 2*lip.std(), color='gray', linestyle='--', alpha=0.4,
               label='mean + 2*std')
    ax.set_xlabel('Mutation Walk Step')
    ax.set_ylabel('Local Lipschitz Constant')
    ax.set_title(f'{name} -- Lipschitz Profile', fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.2)

    # --- Panel C: 2D PCA trajectory ---
    ax = axes[row, 2]
    pca = PCA(n_components=2)
    embs_2d = pca.fit_transform(emb)
    scatter = ax.scatter(embs_2d[:, 0], embs_2d[:, 1], c=steps, cmap='viridis',
                         s=25, alpha=0.8, edgecolors='white', linewidth=0.3)
    ax.plot(embs_2d[:, 0], embs_2d[:, 1], color='gray', alpha=0.3, linewidth=0.8)
    ax.scatter(embs_2d[0, 0], embs_2d[0, 1], c='blue', s=100, marker='s',
               zorder=10, edgecolors='black', linewidth=1.5, label='Wildtype')
    ax.scatter(embs_2d[-1, 0], embs_2d[-1, 1], c='red', s=100, marker='D',
               zorder=10, edgecolors='black', linewidth=1.5, label='Pathogenic')
    if pathogenic_step is not None:
        ax.scatter(embs_2d[pathogenic_step, 0], embs_2d[pathogenic_step, 1],
                   c='red', s=60, marker='x', zorder=10, linewidth=2, label='C61G step')
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    ax.set_title(f'{name} -- Embedding PCA', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)
    plt.colorbar(scatter, ax=ax, label='Walk Step')

model_list = ' vs '.join(embeddings.keys())
plt.suptitle(f'Track B: BRCA1 Single-Point Mutation Walk\n{model_list}',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()

fig_path = f'{RESULTS_DIR}/track_b_brca1_walk_4model.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.savefig(fig_path.replace('.png', '.pdf'), bbox_inches='tight')
print(f'Saved: {fig_path}')
plt.show()

In [ ]:
# Cell: Overlay Lipschitz Comparison (Paper Figure -- All 4 Models)
#
# This is the "Lipschitz Spike" panel: all models overlaid on the same axes.

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# --- Panel 1: Lipschitz profiles overlaid ---
ax = axes[0]
for name in embeddings:
    lip = lipschitz[name]
    norm_steps = np.linspace(0, 1, len(lip))
    color = MODEL_REGISTRY[name]['color']
    label = f'{name} ({MODEL_REGISTRY[name]["arch"]})'
    ax.plot(norm_steps, lip, color=color, linewidth=2, label=label, alpha=0.8)
    ax.fill_between(norm_steps, lip, alpha=0.08, color=color)

if pathogenic_step is not None:
    norm_pathogenic = pathogenic_step / (n_walk - 1)
    ax.axvline(norm_pathogenic, color='black', linestyle='--', alpha=0.5,
               label='C61G mutation')

ax.set_xlabel('Normalized Walk Progress (0=Wildtype, 1=Pathogenic)')
ax.set_ylabel('Local Lipschitz Constant (Cosine)')
ax.set_title('Lipschitz Profile Comparison', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.2)

# --- Panel 2: Cumulative cosine drift overlaid ---
ax = axes[1]
for name in embeddings:
    cos_d = cosine_drift[name]
    color = MODEL_REGISTRY[name]['color']
    ax.plot(np.arange(len(cos_d)), cos_d, color=color, linewidth=2, label=name)

if pathogenic_step is not None:
    ax.axvline(pathogenic_step, color='black', linestyle='--', alpha=0.5,
               label='C61G mutation')

ax.set_xlabel('Mutation Walk Step')
ax.set_ylabel('Cosine Distance from Wildtype')
ax.set_title('Embedding Drift from Wildtype', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.2)

plt.suptitle('Track B: Lipschitz Comparison -- Discrete vs Continuous Architectures',
             fontsize=13, fontweight='bold')
plt.tight_layout()

fig_path = f'{RESULTS_DIR}/track_b_lipschitz_overlay_4model.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.savefig(fig_path.replace('.png', '.pdf'), bbox_inches='tight')
print(f'Saved: {fig_path}')
plt.show()

In [ ]:
# Cell: Cross-Track Comparison (Track A + Track B)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# --- Panel A: Track A (synthetic, if cached) ---
ax = axes[0]
track_a_cache = './results/interpolation_track_a/cache/'
track_a_found = False

for arch, color, label in [('SmallBERT', '#2563EB', 'SmallBERT (Transformer)'),
                            ('SmallMamba', '#16A34A', 'SmallMamba (SSM)'),
                            ('SmallLSTM', '#F59E0B', 'SmallLSTM'),
                            ('SmallStripedHyena', '#7C3AED', 'SmallStripedHyena (Hybrid)')]:
    lip_path = f'{track_a_cache}{arch}_lipschitz_mean.npy'
    try:
        lip = np.load(lip_path)
        alpha_steps = np.linspace(0, 1, len(lip))
        ax.plot(alpha_steps, lip, color=color, linewidth=2, label=label)
        ax.fill_between(alpha_steps, lip, alpha=0.1, color=color)
        track_a_found = True
    except FileNotFoundError:
        pass

if track_a_found:
    ax.set_xlabel('Interpolation alpha (0=Start, 1=End)')
    ax.set_ylabel('Local Lipschitz Constant')
    ax.set_title('Track A: Synthetic Physics\n(Damped Oscillator)', fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.2)
else:
    ax.text(0.5, 0.5, 'Track A results not yet cached.\nRun Track A notebook first.',
            transform=ax.transAxes, ha='center', va='center', fontsize=12, color='gray')
    ax.set_title('Track A: Synthetic Physics\n(not yet available)', fontweight='bold')

# --- Panel B: Track B (biological, all models) ---
ax = axes[1]
for name in embeddings:
    lip = lipschitz[name]
    norm_steps = np.linspace(0, 1, len(lip))
    color = MODEL_REGISTRY[name]['color']
    ax.plot(norm_steps, lip, color=color, linewidth=2,
            label=f'{name} ({MODEL_REGISTRY[name]["type"]})', alpha=0.8)
    ax.fill_between(norm_steps, lip, alpha=0.08, color=color)

if pathogenic_step is not None:
    norm_pathogenic = pathogenic_step / (n_walk - 1)
    ax.axvline(norm_pathogenic, color='red', linestyle='--', alpha=0.5,
               label='C61G mutation')

ax.set_xlabel('Normalized Walk Progress (0=Wildtype, 1=Pathogenic)')
ax.set_ylabel('Local Lipschitz Constant')
ax.set_title('Track B: BRCA1 Mutation Walk\n(Biological, 4 Models)', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.2)

plt.suptitle('Cross-Track Lipschitz Comparison\nSynthetic Physics vs Biological Reality',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()

fig_path = f'{RESULTS_DIR}/cross_track_comparison_4model.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.savefig(fig_path.replace('.png', '.pdf'), bbox_inches='tight')
print(f'Saved: {fig_path}')
plt.show()

In [ ]:
# Cell: 3D PCA Trajectories (The "Viral Image" -- 4 Models)

from mpl_toolkits.mplot3d import Axes3D

n_models = len(embeddings)
fig = plt.figure(figsize=(20, 5 * ((n_models + 1) // 2)))

for idx, name in enumerate(embeddings):
    emb = embeddings[name]
    color = MODEL_REGISTRY[name]['color']

    ax = fig.add_subplot((n_models + 1) // 2, 2, idx + 1, projection='3d')

    pca = PCA(n_components=3)
    embs_3d = pca.fit_transform(emb)
    var_exp = pca.explained_variance_ratio_ * 100

    ax.plot(embs_3d[:, 0], embs_3d[:, 1], embs_3d[:, 2],
            color=color, linewidth=1.5, alpha=0.6)
    ax.scatter(embs_3d[:, 0], embs_3d[:, 1], embs_3d[:, 2],
               c=np.arange(len(embs_3d)), cmap='viridis',
               s=20, alpha=0.8, edgecolors='white', linewidth=0.3)

    ax.scatter(*embs_3d[0], c='blue', s=120, marker='s',
               edgecolors='black', linewidth=1.5, label='Wildtype', zorder=10)
    ax.scatter(*embs_3d[-1], c='red', s=120, marker='D',
               edgecolors='black', linewidth=1.5, label='Pathogenic', zorder=10)

    if pathogenic_step is not None and pathogenic_step < len(embs_3d):
        ax.scatter(*embs_3d[pathogenic_step], c='red', s=80, marker='x',
                   linewidth=2.5, label='C61G', zorder=10)

    ax.set_xlabel(f'PC1 ({var_exp[0]:.1f}%)')
    ax.set_ylabel(f'PC2 ({var_exp[1]:.1f}%)')
    ax.set_zlabel(f'PC3 ({var_exp[2]:.1f}%)')
    ax.set_title(f'{name}\n({MODEL_REGISTRY[name]["arch"]})',
                 fontweight='bold', fontsize=11)
    ax.legend(fontsize=7, loc='upper left')

plt.suptitle('Track B: 3D Embedding Trajectories -- BRCA1 Mutation Walk\n'
             'Discrete Transformers vs Hybrid vs Continuous SSM',
             fontsize=13, fontweight='bold')
plt.tight_layout()

fig_path = f'{RESULTS_DIR}/track_b_3d_trajectories_4model.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.savefig(fig_path.replace('.png', '.pdf'), bbox_inches='tight')
print(f'Saved: {fig_path}')
plt.show()

In [ ]:
# Quantitative Summary Table

import pandas as pd

results = []
for name in embeddings:
    emb = embeddings[name]
    lip = lipschitz[name]
    cos_d = cosine_drift[name]
    info = MODEL_REGISTRY[name]

    threshold = lip.mean() + 2 * lip.std()
    n_spikes = int((lip > threshold).sum())

    path_spike = False
    if pathogenic_step is not None and pathogenic_step <= len(lip):
        path_spike = lip[pathogenic_step - 1] > threshold

    results.append({
        'Model': name,
        'Architecture': info['arch'],
        'Type': info['type'],
        'Embed Dim': emb.shape[1],
        'Cosine Lip (mean)': f'{lip.mean():.6f}',
        'Cosine Lip (max)': f'{lip.max():.6f}',
        'Cosine Lip (std)': f'{lip.std():.6f}',
        'Smoothness': f'{lip.mean() / (lip.max() + 1e-8):.4f}',
        'Spikes (>2std)': n_spikes,
        'C61G Spike': 'YES' if path_spike else 'no',
        'Total Drift': f'{cos_d[-1]:.6f}',
    })

df = pd.DataFrame(results)
print(df.to_string(index=False))

csv_path = f'{RESULTS_DIR}/track_b_summary_4model.csv'
df.to_csv(csv_path, index=False)
print(f'\nSaved: {csv_path}')

# --- Grouped comparison ---
print('\n' + '=' * 70)
print('ARCHITECTURE GROUP COMPARISON')
print('=' * 70)

for group_name, group_type in [('Discrete (Transformers)', 'discrete'),
                                ('Hybrid', 'hybrid'),
                                ('Continuous (SSM)', 'continuous')]:
    group_models = [n for n in embeddings if MODEL_REGISTRY[n]['type'] == group_type]
    if group_models:
        mean_lip = np.mean([lipschitz[n].mean() for n in group_models])
        mean_smooth = np.mean([lipschitz[n].mean() / (lipschitz[n].max() + 1e-8)
                               for n in group_models])
        total_spikes = sum(int((lipschitz[n] > lipschitz[n].mean() + 2*lipschitz[n].std()).sum())
                           for n in group_models)
        print(f'\n{group_name}: {", ".join(group_models)}')
        print(f'  Mean Lipschitz:  {mean_lip:.6f}')
        print(f'  Mean Smoothness: {mean_smooth:.4f}')
        print(f'  Total Spikes:    {total_spikes}')